In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%autosave 0
    
import logging
logging.basicConfig(format='%(asctime)s %(levelname)s:%(message)s', level=logging.WARNING, datefmt='%I:%M:%S')

import os
import sys
import numpy as np
import pandas as pd
import sdata
import uuid
print("sdata v{}".format(sdata.__version__))

Autosave disabled
sdata v0.25.1


In [5]:
import sdata.iolib.json1sqlitestore
from sdata.iolib.json1sqlitestore import JSONSQLiteStore

In [9]:
db_path = os.path.join("/tmp", "json1store.sqlite")
if os.path.exists(db_path):
    print(f"remove {db_path}")
    os.remove(db_path)

with JSONSQLiteStore(db_path) as store:
    # Einfügen
    uid = store.insert({"user": "alice", "score": 42})
    print("Neue ID:", uid)
    uid = store.insert({"user": "alices", "score": 43})
    print("Neue ID:", uid)

    # Lesen
    alice = store.get(uid)
    print("Gelesen:", alice)

    # Batch-Einfüge
    for u in [{"user": "bob", "score": 37}, {"user": "carol", "score": 58}]:
        store.insert(u)

    # Suche nach high scores
    highs = store.find_by("score", ">", 40)
    print("Highscores:", highs)

    # Update (ganzes Objekt)
    if alice:
        alice["score"] = 99
        store.update(uid, alice)

    # Löschen
    store.delete(uid)

    # Alle Datensätze via Iterator
    for rec in store:
        print("Datensatz:", rec)


remove /tmp/json1store.sqlite
Neue ID: 1
Neue ID: 2
Gelesen: {'user': 'alices', 'score': 43}
Highscores: [{'user': 'carol', 'score': 58}, {'user': 'alices', 'score': 43}, {'user': 'alice', 'score': 42}]
Datensatz: {'user': 'alice', 'score': 42}
Datensatz: {'user': 'bob', 'score': 37}
Datensatz: {'user': 'carol', 'score': 58}


In [4]:
import sqlite3

# Verbindung zu einer In-Memory-DB aufbauen
con = sqlite3.connect(":memory:")
con.enable_load_extension(True)
cur = con.execute("PRAGMA compile_options;")

# Alle Compile-Optionen ausgeben und JSON1 herausfiltern
options = [row[0] for row in cur.fetchall()]
print("Compile-Optionen:", options)
print("JSON1 aktiv:", any("JSON1" in opt for opt in options))


Compile-Optionen: ['ATOMIC_INTRINSICS=1', 'COMPILER=gcc-11.2.0', 'DEFAULT_AUTOVACUUM', 'DEFAULT_CACHE_SIZE=-2000', 'DEFAULT_FILE_FORMAT=4', 'DEFAULT_JOURNAL_SIZE_LIMIT=-1', 'DEFAULT_MMAP_SIZE=0', 'DEFAULT_PAGE_SIZE=4096', 'DEFAULT_PCACHE_INITSZ=20', 'DEFAULT_RECURSIVE_TRIGGERS', 'DEFAULT_SECTOR_SIZE=4096', 'DEFAULT_SYNCHRONOUS=2', 'DEFAULT_WAL_AUTOCHECKPOINT=1000', 'DEFAULT_WAL_SYNCHRONOUS=2', 'DEFAULT_WORKER_THREADS=0', 'DIRECT_OVERFLOW_READ', 'ENABLE_COLUMN_METADATA', 'ENABLE_DBSTAT_VTAB', 'ENABLE_FTS3', 'ENABLE_FTS3_TOKENIZER', 'ENABLE_FTS4', 'ENABLE_FTS5', 'ENABLE_GEOPOLY', 'ENABLE_MATH_FUNCTIONS', 'ENABLE_RTREE', 'ENABLE_UNLOCK_NOTIFY', 'MALLOC_SOFT_LIMIT=1024', 'MAX_ATTACHED=10', 'MAX_COLUMN=2000', 'MAX_COMPOUND_SELECT=500', 'MAX_DEFAULT_PAGE_SIZE=8192', 'MAX_EXPR_DEPTH=10000', 'MAX_FUNCTION_ARG=127', 'MAX_LENGTH=1000000000', 'MAX_LIKE_PATTERN_LENGTH=50000', 'MAX_MMAP_SIZE=0x7fff0000', 'MAX_PAGE_COUNT=0xfffffffe', 'MAX_PAGE_SIZE=65536', 'MAX_SQL_LENGTH=1000000000', 'MAX_TRIGGER_D

In [5]:
try:
    con.execute("SELECT json('{\"key\": 1}')")
    print("JSON1-Funktionen verfügbar")
except sqlite3.OperationalError as e:
    print("Keine JSON1-Funktionen:", e)


JSON1-Funktionen verfügbar


In [6]:
import sqlite3
con = sqlite3.connect("meine.db")
con.enable_load_extension(True)
con.load_extension("/pfad/zu/json1")  # unter Windows .dll, unter macOS .dylib
con.enable_load_extension(False)

# Test:
print(con.execute("SELECT json('{\"key\": 1}')").fetchone())


OperationalError: /pfad/zu/json1.so: cannot open shared object file: No such file or directory

In [7]:
import sqlite3, json

con = sqlite3.connect(":memory:")
cur = con.cursor()
cur.execute("CREATE TABLE t(id INTEGER PRIMARY KEY, doc TEXT)")

payload = {"user": "bob", "settings": {"theme": "dark", "lang": "de"}}
cur.execute("INSERT INTO t(doc) VALUES (?)", (json.dumps(payload),))

# Extrahiere das Theme-Feld
cur.execute("SELECT json_extract(doc, '$.settings.theme') FROM t WHERE id = 1")
theme = cur.fetchone()[0]
print(theme)  # -> dark


dark


In [8]:
import sqlite3, json

con = sqlite3.connect(":memory:")
cur = con.cursor()
cur.execute("CREATE TABLE t(id INTEGER PRIMARY KEY, doc TEXT)")

cur.execute("INSERT INTO t(doc) VALUES (?)", (json.dumps({"count": 1}),))

# Erhöhe count auf 2 und setze newKey
cur.execute("""
    UPDATE t
    SET doc = json_set(doc,
        '$.count', json_extract(doc, '$.count') + 1,
        '$.newKey', 'neu'
    )
    WHERE id = 1
""")
con.commit()

cur.execute("SELECT doc FROM t WHERE id = 1")
print(cur.fetchone()[0])
# -> {"count":2,"newKey":"neu"}


{"count":2,"newKey":"neu"}
